# Step 3: Chunking & Embedding into Vector Database

In this notebook we:
1. Load the extracted text files from Step 2
2. Split them into chunks using `RecursiveCharacterTextSplitter`
3. Store the chunks in ChromaDB with embeddings
4. Test retrieval with a sample query

**Embedding model:** Chroma's default — `all-MiniLM-L6-v2` from sentence-transformers.  
Free, runs locally, no API key needed.

## 3.1 Install Dependencies

In [1]:
!pip install langchain-text-splitters chromadb sentence-transformers

   ---------------------------------------- 0.0/557.4 kB ? eta -:--:--
   ------------------ --------------------- 262.1/557.4 kB ? eta -:--:--
   ---------------------------------------- 557.4/557.4 kB 2.4 MB/s  0:00:00
   ---------------------------------------- 0.0/671.1 kB ? eta -:--:--
   ------------------------------- -------- 524.3/671.1 kB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 671.1/671.1 kB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/23.5 MB 4.1 MB/s eta 0:00:06
   -- ------------------------------------- 1.6/23.5 MB 4.4 MB/s eta 0:00:06
   ---- ----------------------------------- 2.9/23.5 MB 4.9 MB/s eta 0:00:05
   -------- ------------------------------- 4.7/23.5 MB 5.8 MB/s eta 0:00:04
   ----------- ---------------------------- 6.8/23.5 MB 6.6 MB/s eta 0:00:03
   ------------ --------------------------- 7.6/23.5 MB 6.8 MB/s eta 0:00:03
   -------------- -

## 3.2 Imports & Configuration

In [2]:
import os
import json
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Paths — should match Step 2 output
EXTRACTED_DIR = os.path.join("data", "extracted_texts")
CHROMA_DB_DIR = os.path.join("data", "chroma_db")
COLLECTION_NAME = "sec_filings"

# Company config — must match previous steps
COMPANIES = [
    ("MSFT",  "10-K", "Microsoft"),
    ("GOOGL", "10-K", "Alphabet (Google)"),
    ("ACN",   "10-K", "Accenture"),
    ("IBM",   "10-K", "IBM"),
    ("CTSH",  "10-K", "Cognizant"),
    ("INFY",  "20-F", "Infosys"),
    ("WIT",   "20-F", "Wipro"),
]

print(f"Extracted texts dir: {os.path.abspath(EXTRACTED_DIR)}")
print(f"ChromaDB will be stored at: {os.path.abspath(CHROMA_DB_DIR)}")

Extracted texts dir: c:\Users\anfas\Downloads\finance report rag\data\extracted_texts
ChromaDB will be stored at: c:\Users\anfas\Downloads\finance report rag\data\chroma_db


## 3.3 Load Extracted Texts

In [3]:
documents = {}  # ticker -> text content

for ticker, form_type, name in COMPANIES:
    filename = f"{ticker}_{form_type}_extracted.txt"
    filepath = os.path.join(EXTRACTED_DIR, filename)
    
    if not os.path.exists(filepath):
        print(f"  ✗ {ticker} — file not found: {filepath}")
        continue
    
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    
    documents[ticker] = {
        'text': text,
        'form_type': form_type,
        'company_name': name,
        'char_count': len(text),
    }
    print(f"  ✓ {ticker} ({name}) — {len(text):,} characters")

total_chars = sum(d['char_count'] for d in documents.values())
print(f"\nLoaded {len(documents)} documents, {total_chars:,} total characters")

  ✓ MSFT (Microsoft) — 369,437 characters
  ✓ GOOGL (Alphabet (Google)) — 401,420 characters
  ✓ ACN (Accenture) — 407,810 characters
  ✓ IBM (IBM) — 213,601 characters
  ✓ CTSH (Cognizant) — 388,128 characters
  ✓ INFY (Infosys) — 1,040,588 characters
  ✓ WIT (Wipro) — 945,558 characters

Loaded 7 documents, 3,766,542 total characters


## 3.4 Chunking

We use `RecursiveCharacterTextSplitter` which tries to keep paragraphs intact,
then falls back to sentences, then words.

**Chunk size = 1000 characters** with **200 character overlap** is a solid starting
point for financial documents. You can experiment with these later.

- Too small (< 500): loses context, retrieval returns fragments
- Too large (> 2000): dilutes relevance, retrieved chunks contain too much noise
- Overlap: ensures important information at chunk boundaries isn't lost

In [4]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = []     # list of chunk text strings
all_metadatas = []  # list of metadata dicts
all_ids = []        # list of unique IDs

for ticker, doc_info in documents.items():
    text = doc_info['text']
    chunks = text_splitter.split_text(text)
    
    for i, chunk in enumerate(chunks):
        chunk_id = f"{ticker}_chunk_{i:04d}"
        metadata = {
            'ticker': ticker,
            'company_name': doc_info['company_name'],
            'form_type': doc_info['form_type'],
            'chunk_index': i,
            'total_chunks': len(chunks),
        }
        
        all_chunks.append(chunk)
        all_metadatas.append(metadata)
        all_ids.append(chunk_id)
    
    print(f"  {ticker}: {len(chunks)} chunks")

print(f"\nTotal chunks created: {len(all_chunks)}")
print(f"Average chunk size: {sum(len(c) for c in all_chunks) / len(all_chunks):.0f} characters")

  MSFT: 583 chunks
  GOOGL: 627 chunks
  ACN: 649 chunks
  IBM: 300 chunks
  CTSH: 616 chunks
  INFY: 1570 chunks
  WIT: 1441 chunks

Total chunks created: 5786
Average chunk size: 697 characters


## 3.5 Preview Some Chunks

Let's look at a few chunks to verify they look reasonable.

In [5]:
# Show 3 sample chunks from different companies
import random
random.seed(42)

# Pick one chunk from each of the first 3 companies
shown = set()
samples = []
for i, meta in enumerate(all_metadatas):
    if meta['ticker'] not in shown and len(samples) < 3:
        samples.append(i)
        shown.add(meta['ticker'])

for idx in samples:
    meta = all_metadatas[idx]
    chunk = all_chunks[idx]
    print(f"=== {meta['ticker']} | Chunk {meta['chunk_index']}/{meta['total_chunks']} ===")
    print(chunk[:500])
    if len(chunk) > 500:
        print("... [truncated]")
    print()

=== MSFT | Chunk 0/583 ===
10-K

=== GOOGL | Chunk 0/627 ===
goog-20251231
FALSE
2025
FY
0001652044
P7Y
P2Y
http://fasb.org/us-gaap/2025#Revenues
http://fasb.org/us-gaap/2025#NonoperatingIncomeExpense
http://fasb.org/us-gaap/2025#Revenues
http://fasb.org/us-gaap/2025#NonoperatingIncomeExpense
http://fasb.org/us-gaap/2025#Revenues
http://fasb.org/us-gaap/2025#NonoperatingIncomeExpense
http://fasb.org/us-gaap/2025#OtherAssetsCurrent
http://fasb.org/us-gaap/2025#OtherAssetsCurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaa
... [truncated]

=== ACN | Chunk 0/649 ===
acn-20250831
FALSE
FALSE
2025
FY
0001467373
July 24, 2026
275
July 24, 2026
275
July 24, 2026
275
July 24, 2026
270
P1Y
P1Y
P1Y
P4Y
http://fasb.org/us-gaap/2025#OtherNonoperatingIncomeExpense
http://fasb.org/us-gaap/2025#OtherNonoperatingIncomeExpense
http://fasb.org/us-gaap/2025#OtherNonoperatingIncomeExpense
http://fasb.org/us-gaap/2025#OtherNonoperatingIncomeExpense
http://fasb.org/us-gaap

## 3.6 Store in ChromaDB

ChromaDB will:
1. Take our text chunks
2. Embed them using `all-MiniLM-L6-v2` (downloaded automatically on first run)
3. Store the embeddings + text + metadata on disk

**Note:** The first run will download the embedding model (~80MB). This is a one-time download.

We use `PersistentClient` so the database survives between notebook sessions.

In [6]:
# Initialize ChromaDB with persistent storage
client = chromadb.PersistentClient(path=CHROMA_DB_DIR)

# Delete existing collection if re-running this notebook
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'")
except Exception:
    pass

# Create fresh collection
# Chroma uses all-MiniLM-L6-v2 as the default embedding function
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # use cosine similarity
)

print(f"Created collection '{COLLECTION_NAME}'")
print(f"Database path: {os.path.abspath(CHROMA_DB_DIR)}")

Created collection 'sec_filings'
Database path: c:\Users\anfas\Downloads\finance report rag\data\chroma_db


In [7]:
# Add chunks to ChromaDB in batches
# Chroma has a limit on batch size, so we add in batches of 500
BATCH_SIZE = 500

total_added = 0
for start in range(0, len(all_chunks), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(all_chunks))
    
    collection.add(
        documents=all_chunks[start:end],
        metadatas=all_metadatas[start:end],
        ids=all_ids[start:end],
    )
    
    total_added += (end - start)
    print(f"  Added batch {start}-{end} ({total_added}/{len(all_chunks)} chunks)")

print(f"\nDone. Collection now has {collection.count()} chunks.")

C:\Users\anfas\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:24<00:00, 3.45MiB/s]


  Added batch 0-500 (500/5786 chunks)
  Added batch 500-1000 (1000/5786 chunks)
  Added batch 1000-1500 (1500/5786 chunks)
  Added batch 1500-2000 (2000/5786 chunks)
  Added batch 2000-2500 (2500/5786 chunks)
  Added batch 2500-3000 (3000/5786 chunks)
  Added batch 3000-3500 (3500/5786 chunks)
  Added batch 3500-4000 (4000/5786 chunks)
  Added batch 4000-4500 (4500/5786 chunks)
  Added batch 4500-5000 (5000/5786 chunks)
  Added batch 5000-5500 (5500/5786 chunks)
  Added batch 5500-5786 (5786/5786 chunks)

Done. Collection now has 5786 chunks.


## 3.7 Test Retrieval

Let's verify the vector database works by querying it with financial questions.

In [8]:
def query_collection(question, n_results=3, ticker_filter=None):
    """
    Query the vector database and return relevant chunks.
    
    Args:
        question: natural language query
        n_results: number of chunks to retrieve
        ticker_filter: optional ticker to filter by (e.g., 'MSFT')
    """
    where_filter = None
    if ticker_filter:
        where_filter = {"ticker": ticker_filter}
    
    results = collection.query(
        query_texts=[question],
        n_results=n_results,
        where=where_filter,
    )
    
    print(f"Query: \"{question}\"")
    if ticker_filter:
        print(f"Filter: {ticker_filter}")
    print(f"Results: {len(results['documents'][0])} chunks\n")
    
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    )):
        print(f"--- Result {i+1} | {meta['ticker']} ({meta['company_name']}) | Distance: {dist:.4f} ---")
        print(doc[:400])
        if len(doc) > 400:
            print("... [truncated]")
        print()
    
    return results

In [9]:
# Test query 1: broad financial question
_ = query_collection("What are the main risk factors?")

Query: "What are the main risk factors?"
Results: 3 chunks

--- Result 1 | ACN (Accenture) | Distance: 0.2478 ---
Item 1A. Risk Factors

--- Result 2 | CTSH (Cognizant) | Distance: 0.3928 ---
Item 1A. Risk Factors
Investing in our common stock involves a high degree of risk. Before making an investment decision, you should carefully consider the risks described below in addition to the other information set forth in this Annual Report on Form 10-K, including "
Part II, Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations

--- Result 3 | MSFT (Microsoft) | Distance: 0.3933 ---
For a description of the risks we face related to regulatory matters, refer to Risk Factors (Part I, Item 1A of this Form 10-K).



In [10]:
# Test query 2: company-specific question
_ = query_collection("What is the total revenue?", ticker_filter="MSFT")

Query: "What is the total revenue?"
Filter: MSFT
Results: 3 chunks

--- Result 1 | MSFT (Microsoft) | Distance: 0.3873 ---
No sales to an individual customer or country other than the United States accounted for more than 10% of revenue for fiscal years 2025, 2024, or 2023.

Revenue, classified by the major geographic areas in which our customers were located, was as follows:

[TABLE]
(In millions) | | |
Year Ended June 30, | | 2025 | | | 2024 | | | 2023 |
United States (a) | | $ | 144,546 | | | $ | 124,704 | | | $ | 
... [truncated]

--- Result 2 | MSFT (Microsoft) | Distance: 0.3959 ---
Unearned revenue comprises mainly unearned revenue related to volume licensing programs, which may include cloud services and Software Assurance ("SA"). Unearned revenue is generally invoiced annually at the beginning of each contract period for multi-year agreements and recognized ratably over the coverage period. Unearned revenue also includes payments for other offerings for which we have been 
...

In [11]:
# Test query 3: topic that spans companies
_ = query_collection("cloud computing strategy and growth", n_results=5)

Query: "cloud computing strategy and growth"
Results: 5 chunks

--- Result 1 | MSFT (Microsoft) | Distance: 0.3862 ---
•
Making our suite of cloud-based services platform-agnostic, available on a wide range of devices and ecosystems, including those of our competitors.
It is uncertain whether our strategies will continue to attract users or generate the revenue required to succeed. If we are not effective in executing organizational and technical changes to increase efficiency and accelerate innovation, or if we f
... [truncated]

--- Result 2 | WIT (Wipro) | Distance: 0.4232 ---
Our ability to compete successfully also depends on our capacity to anticipate and respond effectively to rapid and continuing changes in technology and evolving client requirements, including in areas such as digital, cloud, cybersecurity, artificial intelligence and other emerging technologies. If we do not invest adequately, innovate at an appropriate pace, or successfully adapt our service off
... [truncat

## 3.8 Collection Stats

In [12]:
print(f"Collection: {COLLECTION_NAME}")
print(f"Total chunks stored: {collection.count()}")
print(f"Database location: {os.path.abspath(CHROMA_DB_DIR)}")
print(f"\nChunks per company:")

for ticker, form_type, name in COMPANIES:
    # Count chunks for this ticker
    count = len([m for m in all_metadatas if m['ticker'] == ticker])
    print(f"  {ticker:6s} ({name:20s}): {count} chunks")

print(f"\nChunking config: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"Embedding model: all-MiniLM-L6-v2 (Chroma default)")
print(f"Distance metric: cosine")

Collection: sec_filings
Total chunks stored: 5786
Database location: c:\Users\anfas\Downloads\finance report rag\data\chroma_db

Chunks per company:
  MSFT   (Microsoft           ): 583 chunks
  GOOGL  (Alphabet (Google)   ): 627 chunks
  ACN    (Accenture           ): 649 chunks
  IBM    (IBM                 ): 300 chunks
  CTSH   (Cognizant           ): 616 chunks
  INFY   (Infosys             ): 1570 chunks
  WIT    (Wipro               ): 1441 chunks

Chunking config: size=1000, overlap=200
Embedding model: all-MiniLM-L6-v2 (Chroma default)
Distance metric: cosine


---

## ✅ Step 3 Complete

**What we did:**
- Split extracted text into chunks using `RecursiveCharacterTextSplitter` (1000 chars, 200 overlap)
- Embedded all chunks using `all-MiniLM-L6-v2` (free, local, no API key)
- Stored everything in ChromaDB with persistent storage
- Tested retrieval with sample financial queries

**Output:**
- `data/chroma_db/` — persistent vector database, reusable across sessions

**What to tell interviewers about chunk size:**
- "I started with 1000 chars / 200 overlap based on common recommendations for document QA"
- "Smaller chunks gave better precision but lost context for multi-sentence answers"
- "Larger chunks returned more noise in retrieval results"

**Next step:** `Step4_RAG_Pipeline.ipynb` — connect retrieval to an LLM to answer questions